# AAV2 -- ShallowProfileMLP sur `viab`, brut vs filtre `viab`

Meme archi/boucle d'entrainement que les notebooks `AAV5_SEL_profile_model_*.ipynb` (ShallowProfileMLP,
~27k params, one-hot 140 -> MLP -> 1 scalaire, MSE non ponderee) -- appliquee ici a la cible
`log2_enrichissement_virus_sur_plasmide` (viabilite) sur 2 CSV AAV2 : brut (`AAV2_organoides.csv`) et
trie (`AAV2_organoides_sorted.csv`, filtre `PLASMID_MIN=1` seul -- le cap `RATIO_MAX=100` a ete retire le 2026-09-16, cf. `AAV2_viab_sorting.ipynb` §8, il tronquait par construction toute la queue haute du log2 enrichment reel). Rappel : contrairement a AAV5, cette cible n'est PAS bimodale sur AAV2
(confirme par dithering dans `AAV2_viab_sorting.ipynb`) -- la comparaison ici porte seulement sur la
capacite du MLP a predire le log enrichment, pas sur une structure fit/non-fit a recouvrer.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "Modelization_V2")
sys.path.insert(0, str(_root / "lib"))
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
import jax.numpy as jnp
from flax import nnx
from typing import Optional
import optax
from tqdm.auto import tqdm

from analysisV1 import AA_LABELS, pearson, precision_at_k, plot_topk_recovery

print(f"JAX backend: {jax.default_backend()} -- devices: {jax.devices()}")

L, A = 7, 20

### 1. Chargement -- viab, brut + trie

In [ ]:
VIAB_COL = "log2_enrichissement_virus_sur_plasmide"
use_cols = ["sequence", VIAB_COL]
dtypes = {"sequence": "string", VIAB_COL: "float32"}

datasets = {}
for tag, fname in [
    ("brut", "AAV2_organoides.csv"),
    ("trie (viab)", "AAV2_organoides_sorted.csv"),
]:
    p = Path(fname)
    if not p.exists():
        p = _root / "notebooks/notebooks/AAVs dataset/AAV2" / fname
    assert p.exists(), p
    df = pd.read_csv(p, usecols=use_cols, dtype=dtypes)
    df["sequence"] = df["sequence"].astype("string")
    datasets[tag] = df
    n_finite = int(np.isfinite(df[VIAB_COL].to_numpy()).sum())
    print(f"{tag:14s} {fname:26s} {len(df):,} variants  (viab fini: {n_finite:,})")

lut = np.zeros(256, dtype=np.int64)
for i, aa in enumerate(AA_LABELS):
    lut[ord(aa)] = i

def encode(df):
    return lut[np.frombuffer("".join(df["sequence"]).encode("ascii"), np.uint8)].reshape(len(df), L)

seq_matrices = {tag: encode(df) for tag, df in datasets.items()}

### 2. Architecture -- ShallowProfileMLP (archi standard du projet, pas de variante deep)

In [ ]:
class ShallowProfileMLP(nnx.Module):
    '''Archi standard du projet (Linear+BatchNorm+Dropout+GELU x2, ~27k params).'''

    def __init__(self, input_dim: int, hidden_dims: tuple[int, int] = (128, 64),
                 dropout_rate: float = 0.1, *, rngs: nnx.Rngs):
        h1, h2 = hidden_dims
        self.linear1    = nnx.Linear(input_dim, h1, rngs=rngs)
        self.batchnorm1 = nnx.BatchNorm(h1, use_running_average=False, rngs=rngs)
        self.dropout1   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear2    = nnx.Linear(h1, h2, rngs=rngs)
        self.batchnorm2 = nnx.BatchNorm(h2, use_running_average=False, rngs=rngs)
        self.dropout2   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear3    = nnx.Linear(h2, 1, rngs=rngs)

    def __call__(self, x: jax.Array, *, train: bool, rngs: Optional[nnx.Rngs] = None) -> jax.Array:
        x = self.linear1(x)
        x = self.batchnorm1(x, use_running_average=not train)
        x = self.dropout1(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)
        x = self.linear2(x)
        x = self.batchnorm2(x, use_running_average=not train)
        x = self.dropout2(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)
        return self.linear3(x).squeeze(-1)


_n_shallow = sum(p.size for p in jax.tree.leaves(nnx.state(ShallowProfileMLP(input_dim=L * A, rngs=nnx.Rngs(0)), nnx.Param)))
print(f"ShallowProfileMLP: {_n_shallow:,} params")

### 3. Boucle d'entrainement -- reprise verbatim des notebooks AAV5

In [ ]:
@nnx.jit
def train_step(model, optimizer, x, y, rngs):
    def loss_fn(model, rngs):
        y_pred = model(x, train=True, rngs=rngs)
        return jnp.mean((y_pred - y) ** 2)
    loss, grads = nnx.value_and_grad(loss_fn)(model, rngs)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def eval_step(model, x, y):
    y_pred = model(x, train=False)
    return jnp.mean((y_pred - y) ** 2)


@nnx.jit
def predict_step(model, x):
    return model(x, train=False)


@nnx.scan(in_axes=(nnx.Carry, 0, 0), out_axes=(nnx.Carry, 0))
def train_epoch_scan(carry, xb, yb):
    model, optimizer, rngs = carry
    loss = train_step(model, optimizer, xb, yb, rngs)
    return (model, optimizer, rngs), loss


def split_train_val(X, y, val_frac=0.15, seed=0):
    rng   = np.random.default_rng(seed)
    idx   = rng.permutation(len(X))
    n_val = int(len(X) * val_frac)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]


def train_mlp(model_cls, model_kwargs, X_train, y_train, X_val, y_val,
              epochs=150, batch_size=512, peak_lr=1e-3, final_lr=1e-5,
              weight_decay=0, patience=5, seed=0, verbose=True):
    rngs  = nnx.Rngs(seed)
    model = model_cls(rngs=rngs, **model_kwargs)

    n_train         = X_train.shape[0]
    steps_per_epoch = max(n_train // batch_size, 1)
    total_steps     = steps_per_epoch * epochs

    lr_schedule_fn = optax.warmup_cosine_decay_schedule(
        init_value=0., peak_value=peak_lr,
        warmup_steps=int(total_steps * 0.1),
        decay_steps=int(total_steps * 0.9),
        end_value=final_lr,
    )
    optimizer = nnx.Optimizer(
        model, optax.adamw(learning_rate=lr_schedule_fn, weight_decay=weight_decay), wrt=nnx.Param
    )

    X_train, y_train = jnp.asarray(X_train), jnp.asarray(y_train)
    X_val,   y_val   = jnp.asarray(X_val),   jnp.asarray(y_val)

    shuffle_key = jax.random.key(seed)
    best_val, best_state, bad_epochs = float("inf"), None, 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in tqdm(range(epochs), desc="  epochs", disable=not verbose, leave=False):
        shuffle_key, perm_key = jax.random.split(shuffle_key)
        perm      = jax.random.permutation(perm_key, n_train)
        batch_idx = perm[: steps_per_epoch * batch_size].reshape(steps_per_epoch, batch_size)

        (model, optimizer, rngs), step_losses = train_epoch_scan(
            (model, optimizer, rngs), X_train[batch_idx], y_train[batch_idx]
        )
        train_loss = float(jnp.mean(step_losses))
        val_loss   = float(eval_step(model, X_val, y_val))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val - 1e-5:
            best_val, bad_epochs = val_loss, 0
            best_state = nnx.state(model)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    nnx.update(model, best_state)
    return model, history


def batched_predict(model, X_oh, batch_size=16384):
    chunks = []
    for start in range(0, X_oh.shape[0], batch_size):
        chunks.append(np.asarray(predict_step(model, jnp.asarray(X_oh[start:start + batch_size]))))
    return np.concatenate(chunks)

### 4. Entrainement -- 2 CSV (brut, trie), pleine donnee

`N_CAP` plafonne le nombre de lignes finies utilisees (fit+eval) -- le brut a 2 810 598 lignes
`viab` finies, trop pour tenir un entrainement en temps raisonnable en une seule passe ; sous-echantillonne
a 300 000 lignes finies max (soit 150 000 fit / 150 000 eval, du meme ordre que les volumes
`N_FIT`/`N_EVAL` utilises pour la regression de Potts) -- meme volume applique aux deux CSV pour que
la comparaison brut/trie ne soit pas biaisee par la taille d'echantillon.

In [ ]:
N_CAP = 300_000
RNG = np.random.default_rng(0)

results = {}
for tag, df in datasets.items():
    S = seq_matrices[tag]
    y_all = df[VIAB_COL].to_numpy(np.float64)
    idx_finite = np.flatnonzero(np.isfinite(y_all))
    if len(idx_finite) > N_CAP:
        idx_finite = RNG.choice(idx_finite, N_CAP, replace=False)

    fit_i, eval_i = train_test_split(idx_finite, test_size=0.5, random_state=0)

    oh = lambda idx: np.eye(A, dtype=np.float32)[S[idx]].reshape(len(idx), -1)
    X_fit,  y_fit  = oh(fit_i),  y_all[fit_i]
    X_eval, y_eval = oh(eval_i), y_all[eval_i]
    Xtr, ytr, Xva, yva = split_train_val(X_fit, y_fit, val_frac=0.15, seed=0)

    print(f"[{tag:14s}] train={len(Xtr):,}  val={len(Xva):,}  held-out={len(X_eval):,}", end="  ")
    model, hist = train_mlp(ShallowProfileMLP, dict(input_dim=L * A), Xtr, ytr, Xva, yva, seed=0, verbose=False)
    pred_eval = batched_predict(model, X_eval)
    r = pearson(y_eval, pred_eval)
    print(f"-> {len(hist['train_loss'])} epochs, held-out r={r:+.3f}")

    results[tag] = dict(y_eval=y_eval, pred_eval=pred_eval, r=r, history=hist,
                         n_fit=len(Xtr), n_eval=len(X_eval))

### 5. Log enrichment predit vs reel (held-out)

In [ ]:
tags = list(datasets.keys())
fig, axes = plt.subplots(1, len(tags), figsize=(6.5 * len(tags), 5.5))
for ax, tag in zip(axes, tags):
    res = results[tag]
    y, p = res["y_eval"], res["pred_eval"]
    ax.hexbin(y, p, gridsize=50, bins="log", mincnt=1, cmap="viridis")
    lo, hi = min(y.min(), p.min()), max(y.max(), p.max())
    ax.plot([lo, hi], [lo, hi], "w--", lw=0.9)
    ax.set_title(f"{tag}\nr = {res['r']:+.3f}  (n_fit={res['n_fit']:,}, n_eval={res['n_eval']:,})", fontsize=10)
    ax.set_xlabel("log enrichment reel"); ax.set_ylabel("log enrichment predit")
fig.suptitle("AAV2 viab -- predit vs reel (held-out), ShallowProfileMLP, brut vs trie", y=1.03)
fig.tight_layout()
plt.show()

### 6. Recovery top-k%

In [ ]:
fig, axes = plt.subplots(1, len(tags), figsize=(6.2 * len(tags), 5.8))
for ax, tag in zip(axes, tags):
    res = results[tag]
    plot_topk_recovery(res["y_eval"], res["pred_eval"], k_frac=0.10,
                        xlabel="log enrichment reel", ylabel="log enrichment predit",
                        title=tag, ax=ax)
    ax.legend(fontsize=7)
fig.suptitle("AAV2 viab -- top-10% recovery (held-out)", y=1.03)
fig.tight_layout()
plt.show()

rows = []
for tag in tags:
    res = results[tag]
    row = {"CSV": tag, "n_fit": res["n_fit"], "n_eval": res["n_eval"], "Pearson r": round(res["r"], 3)}
    for frac in (0.01, 0.05, 0.10, 0.20):
        row[f"top-{int(frac*100)}%"] = round(precision_at_k(res["y_eval"], res["pred_eval"], k_frac=frac), 3)
    rows.append(row)
summary = pd.DataFrame(rows)
summary

### 7. Delta vs brut

In [ ]:
base = summary[summary["CSV"] == "brut"].iloc[0]
metrics = ["Pearson r", "top-1%", "top-5%", "top-10%", "top-20%"]
delta = summary[summary["CSV"] != "brut"][["CSV"] + metrics].copy()
for m in metrics:
    delta[m] = (delta[m] - base[m]).round(3)
delta.columns = ["CSV"] + [f"Delta {m} (vs brut)" for m in metrics]
delta

### 8. Notes

- Meme archi/boucle que les notebooks `AAV5_SEL_profile_model_*.ipynb` (ShallowProfileMLP, MSE non
  ponderee) -- aucune adaptation specifique a AAV2 au-dela du choix de la cible et des CSV.
- `N_CAP=300 000` (150k fit / 150k eval) applique identiquement aux deux CSV pour isoler l'effet du
  filtre `viab` de celui du volume de donnees.
- Rappel `AAV2_viab_sorting.ipynb` : la regression de Potts (methode lineaire, 8541 features) donnait
  un held-out r=+0.266 sur le CSV trie -- comparer au r du MLP ici indique si le MLP apporte un
  gain de debruitage/interaction au-dela du modele Potts lineaire, comme observe sur AAV5/AAV9.